In [3]:
"""
실시간 심박수/호흡수 이상치 탐지 데모 (가짜 데이터)

- River의 HalfSpaceTrees(온라인 Isolation Forest 계열)로 스트림을 실시간 판정
- 초반 WARMUP 구간은 학습만 하고 판정은 보류
- 원시값 + 최근 평균 대비 변화량(diff) 특징을 함께 사용
- 중간중간 이상치를 일부러 섞어서 탐지되는지 확인
"""

import random
import time
from collections import deque

from river import anomaly

# ---- 설정 ----------------------------------------------------------
WARMUP = 200          # 이 샘플 수까지는 학습만, 판정 안 함
THRESHOLD = 0.7       # 이상치 점수 임계값 (0~1, 높을수록 이상)
WINDOW = 10           # 변화량 특징을 위한 최근 값 개수
SLEEP = 0.0           # 실제 실시간처럼 보려면 1.0(초)로. 데모는 0.

random.seed(42)

# ---- 가짜 데이터 스트림 --------------------------------------------
def fake_stream(n=1000):
    """정상값(정규분포) + 5% 확률로 이상치를 섞어 내보낸다."""
    for i in range(n):
        if random.random() < 0.05 and i > WARMUP:
            # 이상치: 심박수/호흡수 급등 또는 급락
            hr = random.choice([random.gauss(150, 5), random.gauss(40, 3)])
            rr = random.choice([random.gauss(35, 3), random.gauss(6, 1)])
            is_true_anomaly = True
        else:
            # 정상: 안정된 기준값 주변
            hr = random.gauss(72, 3)
            rr = random.gauss(16, 1)
            is_true_anomaly = False
        yield {"hr": hr, "rr": rr}, is_true_anomaly


# ---- 특징 만들기: 원시값 + 최근 평균 대비 변화량 ------------------
hr_hist = deque(maxlen=WINDOW)
rr_hist = deque(maxlen=WINDOW)

def make_features(x):
    hr, rr = x["hr"], x["rr"]
    hr_mean = sum(hr_hist) / len(hr_hist) if hr_hist else hr
    rr_mean = sum(rr_hist) / len(rr_hist) if rr_hist else rr
    feats = {
        "hr": hr,
        "rr": rr,
        "hr_diff": hr - hr_mean,   # 최근 평균 대비 급변량
        "rr_diff": rr - rr_mean,
    }
    hr_hist.append(hr)
    rr_hist.append(rr)
    return feats


# ---- 실시간 루프 ---------------------------------------------------
def main():
    model = anomaly.HalfSpaceTrees(n_trees=25, height=15, window_size=250)

    n_seen = 0
    detected = 0
    true_positive = 0
    false_positive = 0

    for x, is_true in fake_stream(1000):
        feats = make_features(x)
        score = model.score_one(feats)
        model.learn_one(feats)
        n_seen += 1

        if n_seen <= WARMUP:
            continue  # 워밍업: 판정 보류

        if score > THRESHOLD:
            detected += 1
            tag = "정탐" if is_true else "오탐"
            if is_true:
                true_positive += 1
            else:
                false_positive += 1
            print(f"[{n_seen:4d}] 이상치 감지 "
                  f"hr={x['hr']:6.1f} rr={x['rr']:5.1f} "
                  f"score={score:.3f} ({tag})")

        if SLEEP:
            time.sleep(SLEEP)

    print("\n--- 요약 ---")
    print(f"총 샘플: {n_seen}")
    print(f"감지된 이상치: {detected}건 (정탐 {true_positive}, 오탐 {false_positive})")


if __name__ == "__main__":
    main()

[ 251] 이상치 감지 hr=  76.4 rr= 15.9 score=0.794 (오탐)
[ 252] 이상치 감지 hr=  70.3 rr= 16.3 score=0.716 (오탐)
[ 253] 이상치 감지 hr=  72.2 rr= 14.6 score=0.956 (오탐)
[ 254] 이상치 감지 hr=  73.1 rr= 17.2 score=0.999 (오탐)
[ 255] 이상치 감지 hr=  67.7 rr= 16.6 score=0.995 (오탐)
[ 256] 이상치 감지 hr=  76.1 rr= 15.8 score=0.794 (오탐)
[ 257] 이상치 감지 hr=  74.1 rr= 17.1 score=0.986 (오탐)
[ 258] 이상치 감지 hr=  79.8 rr= 17.7 score=0.985 (오탐)
[ 259] 이상치 감지 hr=  70.5 rr= 16.1 score=0.716 (오탐)
[ 260] 이상치 감지 hr=  67.2 rr= 16.0 score=0.716 (오탐)
[ 261] 이상치 감지 hr=  71.2 rr= 17.9 score=0.886 (오탐)
[ 262] 이상치 감지 hr=  72.1 rr= 17.3 score=0.998 (오탐)
[ 263] 이상치 감지 hr=  75.2 rr= 15.4 score=0.794 (오탐)
[ 264] 이상치 감지 hr=  73.6 rr= 15.4 score=0.959 (오탐)
[ 265] 이상치 감지 hr=  66.0 rr= 17.2 score=0.999 (오탐)
[ 266] 이상치 감지 hr=  73.0 rr= 15.3 score=0.998 (오탐)
[ 267] 이상치 감지 hr=  62.5 rr= 14.1 score=0.716 (오탐)
[ 268] 이상치 감지 hr=  74.2 rr= 17.0 score=0.990 (오탐)
[ 269] 이상치 감지 hr=  68.4 rr= 15.6 score=0.716 (오탐)
[ 270] 이상치 감지 hr=  71.2 rr= 16.6 score=0.999 (오탐)
